In [2]:
print('hello world')

hello world


In [ ]:
import os, json, time, re, textwrap
from pageindex import PageIndexClient
from langchain_groq import ChatGroq
from constants import Groq_API_KEY, Pageindex_API_KEY

c:\Users\vrush\works_code\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [ ]:
os.environ["GROQ_API_KEY"] = Groq_API_KEY
os.environ["PAGEINDEX_API_KEY"] = Pageindex_API_KEY

In [10]:
print("PageIndex key loaded:", "✅" if Pageindex_API_KEY else "❌ Missing!")
print("Groq key loaded:   ", "✅" if Groq_API_KEY    else "❌ Missing!")

PageIndex key loaded: ✅
Groq key loaded:    ✅


In [124]:
file_path=r'C:\Users\vrush\works_code\vecless_rag\machine_learning.pdf'

In [12]:
pi_client= PageIndexClient(api_key=Pageindex_API_KEY)

### documnet upload id and the free token are max 200
##### means 1 page = 1 token

In [ ]:
result = pi_client.submit_document(file_path)

In [137]:
##  pi-cmnqcuwaf050e01pgzep2poqx  ##  Alwasys remeber to check the status of the document processing before querying it. It can take a few minutes depending on the document size.

In [129]:
doc_id = result['doc_id']
print("Document submitted. ID:", doc_id)

Document submitted. ID: pi-cmnqcuwaf050e01pgzep2poqx


### Call the data uploaded document with the doc_id to check the status and get the content

In [15]:
doc_id='pi-cmnqcuwaf050e01pgzep2poqx'

In [16]:
status_result = pi_client.get_document(doc_id)
status= status_result['status']
# print("Document processing status:", status_result['status'])

In [17]:
if status == "completed":
    print("Document processing completed. You can now query the document.")
elif status == "processing":
    print("Document is still processing. Please check back later.")
else:    
    print("Document is still processing. Please check back later.")

Document processing completed. You can now query the document.


 ### ispect the tree structure of the document

In [18]:
tree_result = pi_client.get_tree(doc_id,node_summary=True)
print("Document tree structure:", json.dumps(tree_result, indent=2))

Document tree structure: {
  "doc_id": "pi-cmnqcuwaf050e01pgzep2poqx",
  "status": "completed",
  "retrieval_ready": true,
  "result": [
    {
      "title": "Machine Learning",
      "node_id": "0000",
      "page_index": 1,
      "summary": "# Machine Learning\n",
      "text": "# Machine Learning\n"
    },
    {
      "title": "Basic Concepts",
      "node_id": "0001",
      "page_index": 1,
      "summary": "# Basic Concepts\n\n![img-0.jpeg](img-0.jpeg)\n\n![img-1.jpeg](img-1.jpeg)\n",
      "text": "# Basic Concepts\n\n![img-0.jpeg](img-0.jpeg)\n\n![img-1.jpeg](img-1.jpeg)\n"
    },
    {
      "title": "Terminology",
      "node_id": "0002",
      "page_index": 2,
      "summary": "# Terminology\n\nMachine Learning, Data Science, Data Mining, Data Analysis, Statistical Learning, Knowledge Discovery in Databases, Pattern Discovery.\n\n![img-2.jpeg](img-2.jpeg)\n",
      "text": "# Terminology\n\nMachine Learning, Data Science, Data Mining, Data Analysis, Statistical Learning, Know

In [19]:
page_index_tree= tree_result.get('result', [])
print(F'total pagees:', len(page_index_tree))

total pagees: 28


### Check document status with all the details title, summary, etc

In [20]:
for page in page_index_tree:
    print(f"Page : {page['page_index']}, title: {page['title']}, nodes: {len(page.get('nodes', []))}")
    if 'nodes' in page:
        for node in page['nodes']:
            print(f"  Node ID: {node['node_id']}, title: {node['title']}")

Page : 1, title: Machine Learning, nodes: 0
Page : 1, title: Basic Concepts, nodes: 0
Page : 2, title: Terminology, nodes: 0
Page : 3, title: Data everywhere!, nodes: 0
Page : 4, title: Data types, nodes: 0
Page : 5, title: Smile, we are 'DATAFIED'!, nodes: 0
Page : 6, title: The Data Science process, nodes: 0
Page : 7, title: Applications of ML, nodes: 0
Page : 9, title: Interdisciplinary field, nodes: 0
Page : 10, title: ML versus Statistics, nodes: 2
  Node ID: 0010, title: Statistics:
  Node ID: 0011, title: Machine Learning:
Page : 13, title: Supervised vs. Unsupervised, nodes: 0
Page : 16, title: Unsupervised Learning, nodes: 0
Page : 21, title: Supervised learning, nodes: 2
  Node ID: 0015, title: Classification:
  Node ID: 0016, title: Non linear classification
Page : 31, title: Training and Testing, nodes: 0
Page : 33, title: K-nearest neighbors, nodes: 2
  Node ID: 0019, title: Training algorithm:
  Node ID: 0020, title: Classification algorithm:
Page : 43, title: Application

In [21]:
total_nodes = sum(len(page.get('nodes', [])) for page in page_index_tree)
print(f"Total nodes in the document: {total_nodes}")

Total nodes in the document: 6


### vector rag retrieval

In [30]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7,
    max_tokens=500,
)

### take what data we want from the page index tree and create a prompt for the LLM to answer the question.

In [23]:
out_found=[]
for n in page_index_tree:
    entry={
        'node_id': n['node_id'],
        'title': n['title'],
        'page_index': n['page_index'],
        'summary': n.get('summary', '')
    }
    if n.get('nodes'):
        entry['children_nodes'] = n['nodes']
    out_found.append(entry)

# print(json.dumps(out_found, indent=2))

In [24]:
def Model_prediction(prompt_ask, out_found):
    prompt = f'''  
    You are given a query and a document's tree structure (like a Table of Contents).
    Your task: identify which node IDs most likely contain the answer to the query.
    Think step-by-step about which sections are relevant.

    Query: {prompt_ask}

    Document Tree:
    {json.dumps(out_found, indent=2)}

    Reply ONLY in this exact JSON format:
    {{
        "thinking": "Your step-by-step reasoning about which nodes are relevant",
        "relevant_node_ids": ["list", "of", "node_ids", "most", "likely", "to", "contain", "the", "answer"]
        "titles": ["corresponding", "titles", "of", "the", "relevant", "nodes"]
        "page_indices": ["corresponding", "page", "indices", "of", "the", "relevant", "nodes"]
    }}

    only provide the JSON response without any additional text or explanation.
    '''

    response1 = llm.invoke(prompt)
    print("Model response:", response1.content)

    return response1.content

In [31]:
prompt_ask = "how does machine learning work?"
answer = Model_prediction(prompt_ask, out_found)
json_str = re.search(r'\{.*\}', answer, re.DOTALL).group()
data = json.loads(json_str)

Model response: {
    "thinking": "The query asks for an explanation of how machine learning works. Relevant sections are those that define machine learning, describe its basic concepts, and outline the learning process (training, testing, supervised vs unsupervised). Nodes covering the overall overview (0000), basic concepts (0001), the formal definition and list of techniques (0011), the distinction between supervised and unsupervised learning (0012), a deeper look at supervised learning (0014), and the training/testing workflow (0017) are most likely to contain the answer.",
    "relevant_node_ids": ["0000", "0001", "0011", "0012", "0014", "0017"],
    "titles": ["Machine Learning", "Basic Concepts", "Machine Learning:", "Supervised vs. Unsupervised", "Supervised learning", "Training and Testing"],
    "page_indices": ["1", "1", "10", "13", "21", "31"]
}


In [32]:
print("\n" + "="*60)
print(f"🔍 QUERY: {prompt_ask}")
print("="*60)

print("\n🧠 MODEL THINKING:")
print("-"*60)
print(textwrap.fill(data['thinking'], width=190))

print("\n📌 RELEVANT SECTIONS:")
print("-"*60)

for i, (nid, title, page) in enumerate(zip(
        data['relevant_node_ids'],
        data['titles'],
        data['page_indices']
    ), start=1):
    
    print(f"{i}. 📄 Title       : {title}")
    print(f"   🆔 Node ID    : {nid}")
    print(f"   📑 Page Index : {page}")
    print("-"*60)


🔍 QUERY: how does machine learning work?

🧠 MODEL THINKING:
------------------------------------------------------------
The query asks for an explanation of how machine learning works. Relevant sections are those that define machine learning, describe its basic concepts, and outline the learning process
(training, testing, supervised vs unsupervised). Nodes covering the overall overview (0000), basic concepts (0001), the formal definition and list of techniques (0011), the distinction
between supervised and unsupervised learning (0012), a deeper look at supervised learning (0014), and the training/testing workflow (0017) are most likely to contain the answer.

📌 RELEVANT SECTIONS:
------------------------------------------------------------
1. 📄 Title       : Machine Learning
   🆔 Node ID    : 0000
   📑 Page Index : 1
------------------------------------------------------------
2. 📄 Title       : Basic Concepts
   🆔 Node ID    : 0001
   📑 Page Index : 1
------------------------------

## THE END